In [29]:
pip install hmmlearn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.0/166.0 kB 5.3 MB/s eta 0:00:00


In [30]:
import numpy as np
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import os
from dataclasses import dataclass, field, asdict
from typing import Optional, Sequence, Union, List, Dict, Any, Tuple
import numpy as np
from statsmodels.tsa.ar_model import AutoReg
from sklearn.linear_model import LinearRegression
import cvxpy as cp
import scipy.stats as stats
from hmmlearn.hmm import GaussianHMM

In [32]:
@dataclass
class Universe:
  Bonds:List[str]
  Futures:List[str]
  Commodities:List[str]
  High_Beta:List[str]
  High_Yield:List[str]
  Sat_Defensive:List[str]

@dataclass
class OptimParams:
  es_prct: float
  turnover_penalty: float
  risk_penalty: float
  tail_penalty: float

@dataclass
class FilterParams:
  corr_threshold:float=0.8
  rfr:float=0.003,
  max_r2:float=0.15,
  min_sharpe:float=0.30,
  min_iur:float=0.70,
  T:int=252
  max_equity_corr: float = 0.35,
  min_cpi_corr: float = 0.20,
  max_beta_std: float = 0.25

@dataclass
class HMMParams:
    n_components: int = 2
    n_iter: int = 50
    tol: float = 1e-4
    init_params: str = "stmc"
    covariance_type: str = "diag"


In [25]:
class DataStore:
  def __init__(self, debug:bool=False, **kwargs):
    super().__init__(
        debug=debug,
        **kwargs
    )
    self.debug = debug

  def _get_data(
      self,
      universe:dict,
      start:str,
      end:str,
      interval:str="1d",
      benchmark:str="^GSPC"
  ):
    tickers = list(universe.values())
    self.benchmark_ticker = benchmark
    df_path = f"portfolio_{start}_{end}.parquet"

    if not os.path.exists(df_path):
      tickers.append(benchmark)
      df = yf.download(tickers, start, end, interval, group_by="tickers")

      df.to_parquet(df_path)

    else:
      df = pd.read_parquet(df_path)

    bench_data = df["^GSPC"]["Close"]
    data_raw = df.drop(columns=["^GSPC"])

    benchmark = bench_data.pct_change().dropna()
    self.universe = data_raw.columns

    return data_raw, benchmark

  def plot_data(self):
    (np.cumsum(self.returns_raw * 100, axis=0) + 100).plot(figsize=(15, 10))
    plt.show()

  def plot_benchmark(self):
    (np.cumsum(self.benchmark * 100, axis=0) + 100).plot(figsize=(15, 10))
    plt.show()

In [26]:
class Filter:
  def __init__(self, debug:bool=False, **kwargs):
    super().__init__(
        debug=debug,
        **kwargs
    )
    self.debug = debug

  def corr_filter(self, returns):
    corr_matrix = returns.corr()
    std = returns.std()

    sharpe = (np.mean(returns)-self.rf) / std

    drop = []
    for ticker in returns.columns:
      if ticker in drop:
        continue
      ticker_idx = returns.columns.get_loc(ticker)

      for i in returns.columns:
        if ticker == i or i in drop:
          continue

        i_idx = returns.columns.get_loc(i)
        if corr_matrix.iloc[ticker_idx, i_idx] > self.corr_threshold:
          if self.debug:
            print(corr_matrix.iloc[ticker_idx, i_idx])
          if sharpe[ticker] > sharpe[i]:
            drop.append(i)
          else:
            drop.append(ticker)

    tmp = returns.drop(columns=drop)

    return returns.columns

  def abs_return_unq_filter(
    self,
    returns:pd.DataFrame,
    benchmark:pd.Series,
    rfr:float=0.003,
    max_r2:float=0.15,
    min_sharpe:float=0.30,
    min_iur:float=0.70,
    T:int=252
  )->list:
    model, X, y = self._run_regression(
        [returns, benchmark]
    )
    beta = model.coef_[0]
    r2 = model.score(X, y)

    pred = model.predict(X)
    residuals = y - pred

    iur = np.var(residuals) / np.var(y) if np.var(y) > 0 else 0

    ann_res_mean = np.mean(residuals) * 252
    ann_res_std = np.std(residuals) * np.sqrt(252)
    residual_sharpe = ann_res_mean / ann_res_std if ann_res_std > 0 else -np.inf

    passed_r2 = r2 <= max_r2
    passed_iur = iur >= min_iur
    passed_sharpe = residual_sharpe >= min_sharpe

    is_qualified = passed_r2 and passed_iur and passed_sharpe

    return is_qualified

  def _run_regression(self, asset_list):
    df = pd.concat(
      asset_list,
      axis=1
    ).dropna()
    df.columns = ['asset', 'benchmark']

    X = df[['benchmark']].values
    y = df['asset'].values

    model = LinearRegression().fit(X, y)

    return model, X, y

  def commodity_factor_filter(
      asset_returns: pd.Series,
      market_returns: pd.Series,
      cpi_surprises: pd.Series,
      max_equity_corr: float = 0.35,
      min_cpi_corr: float = 0.20,
      max_beta_std: float = 0.25
  ) -> list:
    df = pd.concat([asset_returns, market_returns, cpi_surprises], axis=1).dropna()
    df.columns = ['Asset', 'Market', 'CPI_Surprise']

    equity_corr = df['Asset'].corr(df['Market'])
    pass_equity = abs(equity_corr) <= max_equity_corr

    cpi_corr = df['Asset'].corr(df['CPI_Surprise'])
    pass_cpi = cpi_corr >= min_cpi_corr

    rolling_cov = df['Asset'].rolling(60).cov(df['Market'])
    rolling_var = df['Market'].rolling(60).var()
    rolling_beta = (rolling_cov / rolling_var).dropna()
    beta_std = rolling_beta.std()
    pass_beta = beta_std <= max_beta_std

    is_qualified = pass_equity and pass_cpi and pass_beta

    return is_qualified


  def high_beta_filter(
    self,
    returns:pd.DataFrame,
    benchmark:pd.DataFrame,
    min_sector_beta:float=1.2
  ):
    model, X, y = self._run_regression(
        [returns, benchmark]
    )
    beta = model.coef_[0]
    passed_beta = beta > min_sector_beta

    return passed_beta

  def downside_capture_filter(self, returns, benchmark):
    down_mask = returns < 0

    if not down_mask.any():
        return 1.0

    asset_down_compound = np.prod(1 + returns[down_mask]) - 1
    bench_down_compound = np.prod(1 + benchmark[down_mask]) - 1

    return asset_down_compound / bench_down_compound


In [33]:
class HMMClassifier:
  def __init__(self, hmm_params, debug=False, **kwargs):
    self.debug = debug
    hmm_params = asdict(hmm_params) if hasattr(hmm_params, '__dataclass_fields__') else hmm_params
    self.hmm = GaussianHMM(**hmm_params)
    self.trained = False

  def fit(self, X):
    self.trained = True
    return self.hmm.fit(X)

  def get_regime_probs(self, X):
    if not self.trained:
      raise ValueError("HMM Classifier not trained")

    return self.hmm.predict_proba(X)

In [34]:
class RegimeClassifier(HMMClassifier):
  def __init__(self, hmm_params, debug=False, **kwargs):
    super().__init__(hmm_params=hmm_params, debug=debug, **kwargs)
    self.s_mean = None
    self.s_std = None
    self.eff_dim_mean = None
    self.eff_dim_std = None
    self.vol_mean = None
    self.vol_std = None

  def _apply_ledoit_wolf(self, r_norm, gram_sample, T, n_assets):
    mean_mkt_corr = (np.sum(gram_sample) - n_assets) / (n_assets * (n_assets - 1))
    target = np.full((n_assets, n_assets), mean_mkt_corr)
    np.fill_diagonal(target, 1.0)

    noise_mtx = 0.0
    for t in range(T):
      r_t = r_norm[t, :].reshape(-1, 1)
      sample_t_mtx = r_t @ r_t.T
      noise_mtx += np.sum((sample_t_mtx - gram_sample) ** 2)

    noise = noise_mtx / (T ** 2)
    dist  = np.sum((gram_sample - target) ** 2)

    if dist == 0:
      return gram_sample

    shrinkage_intensity = np.clip(noise / dist, 0.0, 1.0)
    return shrinkage_intensity * target + (1 - shrinkage_intensity) * gram_sample

  def calculate_spec_ent(self, eig_vals):
    regime_probs = eig_vals / np.sum(eig_vals)
    regime_probs = np.clip(regime_probs, 1e-12, 1.0)
    return -np.sum(regime_probs * np.log(regime_probs))

  def _get_eff_dim(self, r_norm, T):
    G = (1 / T) * (r_norm.T @ r_norm)
    n_assets = G.shape[0]
    G_cond   = self._apply_ledoit_wolf(r_norm, G, T, n_assets)
    eig_vals = np.clip(np.linalg.eigvalsh(G_cond), a_min=1e-12, a_max=None)
    spec_ent = self.calculate_spec_ent(eig_vals)
    return np.exp(spec_ent)

  def garman_klass_vol(self, market, lambda_param=0.94):
    ln_CO = np.log(market["Close"] / market["Open"])
    ln_HL = np.log(market["High"] / market["Low"])
    gk_var = 0.5 * (ln_HL ** 2) - (2 * np.log(2) - 1) * (ln_CO ** 2)

    T = len(gk_var)
    smoothed_variance    = np.zeros(T)
    smoothed_variance[0] = gk_var.iloc[0]
    for t in range(1, T):
      smoothed_variance[t] = (lambda_param * smoothed_variance[t - 1]) + ((1 - lambda_param) * gk_var.iloc[t])
    return np.sqrt(smoothed_variance * 252)

  def _get_rolling_eff_dim(self, r_norm, lookback: int = 60):
      T, n_assets = r_norm.shape
      eff_dim_series = np.full(T, np.nan)
      min_obs = max(10, n_assets + 2)

      for t in range(T):
        start = max(0, t - lookback + 1)
        window = r_norm[start:t + 1, :]
        if window.shape[0] < min_obs:
          continue
        eff_dim_series[t] = self._get_eff_dim(window, window.shape[0])

      valid_mask = ~np.isnan(eff_dim_series)
      if not valid_mask.any():
        raise ValueError("Not enough observations to compute effective dimension.")

      first_valid = np.argmax(valid_mask)
      eff_dim_series[:first_valid] = eff_dim_series[first_valid]
      return eff_dim_series

  def fit_transform_features(self, sector_returns, market_ohlc, eff_dim_lookback=60):
    s_r_log = np.log(1.0 + sector_returns).values

    self.s_mean = np.mean(s_r_log, axis=0)
    self.s_std = np.std(s_r_log, axis=0, ddof=0)
    self.s_std[self.s_std == 0] = 1.0

    s_r_norm = (s_r_log - self.s_mean) / self.s_std
    eff_dim_series = self._get_rolling_eff_dim(s_r_norm, lookback=eff_dim_lookback)
    vol = self.garman_klass_vol(market_ohlc)

    self.eff_dim_mean, self.eff_dim_std = np.mean(eff_dim_series), np.std(eff_dim_series, ddof=0)
    self.vol_mean, self.vol_std = np.mean(vol), np.std(vol, ddof=0)

    eff_dim_norm = (eff_dim_series - self.eff_dim_mean) / (self.eff_dim_std if self.eff_dim_std > 0 else 1.0)
    vol_norm = (vol - self.vol_mean) / (self.vol_std if self.vol_std > 0 else 1.0)

    return np.column_stack((eff_dim_norm, vol_norm))

  def transform_features(self, sector_returns, market_ohlc, eff_dim_lookback=60):
    if self.s_mean is None:
        raise ValueError("Classifier must be trained before calling transform.")

    s_r_log = np.log(1.0 + sector_returns).values
    s_r_norm = (s_r_log - self.s_mean) / self.s_std
    eff_dim_series = self._get_rolling_eff_dim(s_r_norm, lookback=eff_dim_lookback)
    vol = self.garman_klass_vol(market_ohlc)

    eff_dim_norm = (eff_dim_series - self.eff_dim_mean) / (self.eff_dim_std if self.eff_dim_std > 0 else 1.0)
    vol_norm = (vol - self.vol_mean) / (self.vol_std if self.vol_std > 0 else 1.0)

    return np.column_stack((eff_dim_norm, vol_norm))

  def train_classifier(self, returns, market):
    X = self.fit_transform_features(returns, market)
    self.fit(X)

  def get_regime_probs(self, returns, market):
    X = self.transform_features(returns, market)
    return super().get_regime_probs(X)

  def score_bic(self, X):
    T, n_features  = X.shape
    n_components   = self.hmm.n_components
    log_likelihood = self.hmm.score(X) * T
    k = n_components * (n_components - 1) + 2 * (n_components * n_features)
    return k * np.log(T) - 2 * log_likelihood

  def expected_regime_duration(self):
    diag = np.clip(np.diag(self.hmm.transmat_), 1e-10, 1.0 - 1e-4)
    return 1.0 / (1.0 - diag)

In [35]:
class Optimizer:
  def __init__(
      self,
      optim_params,
      debug:bool=False,
      **kwargs
  ):
    self.debug = debug
    self.es_prct = optim_params.es_prct
    self.turnover_penalty = optim_params.turnover_penalty
    self.risk_pen = optim_params.risk_penalty
    self.tail_penalty = optim_params.tail_penalty

  def optimize_lambda(self, p, A, b, k_eq, k_ineq):
    constraints = []
    K = k_eq + k_ineq
    lambda_var = cp.Variable(K)

    if k_ineq > 0:
        constraints.append(lambda_var[k_eq:] >= 0)

    exp_terms = -lambda_var @ A + np.log(p)
    obj_fn = cp.log_sum_exp(exp_terms) + lambda_var @ b

    prob = cp.Problem(cp.Minimize(obj_fn), constraints)
    prob.solve(solver=cp.ECOS)

    if prob.status not in ["optimal", "optimal_inaccurate"]:
        raise RuntimeError(f"Optimization failed. Status: {prob.status}")

    return lambda_var.value

  def optimize_w(self, returns, mean, covariance, w_prev=None):
    R = returns.values
    S, N = R.shape
    w = cp.Variable(N)
    u = cp.Variable(S)
    zeta = cp.Variable()

    mu = mean.values
    cov = covariance.values

    es = zeta + (1 / ((1-self.es_prct) * S)) * cp.sum(u)

    constraints = [
        u >= -R @ w - zeta,
        u >= 0,
        cp.sum(w) == 1,
        w >= 0.0,
        w <= 0.40
    ]

    ex_r = mu @ w

    if w_prev is not None:
      turnover_penalty = self.turnover_penalty * cp.sum((w - w_prev)**2)

    else:
      turnover_penalty = 0

    risk_term = cp.quad_form(w, cov)
    obj_fn = cp.Maximize(ex_r - risk_term - es - turnover_penalty)

    prob = cp.Problem(obj_fn, constraints)
    prob.solve(solver=cp.CLARABEL)

    if prob.status not in ["optimal", "optimal_inaccurate"]:
        raise ValueError(f"GMV optimization failed: {prob.status}")

    return w.value



In [36]:
class Portfolio(DataStore):
  def __init__(self, debug:bool=False, **kwargs):
    super().__init__(
        debug=debug,
        **kwargs
    )
    self.debug = debug

  def get_data(self, tickers, start, end, benchmark="^GSPC"):
    data, benchmark = self._get_data(
        tickers,
        benchmark=benchmark
    )

    cpi = self.get_cpi(start, end)

    return data, benchmark, cpi

  def filter_universe(
    self,
    universe:dict,
    data:pd.DataFrame,
    benchmark:pd.DataFrame,
    cpi:pd.DataFrame,
    filter_params:dict
  ):
    returns = {}
    for ticker in data.columns:
      returns[ticker] = data[ticker]["Close"].pct_change().dropna()

    returns = pd.DataFrame(returns)

  def solve_entropy_pooling(
    self,
    R,
    views=None,
    p=None,
    window=1
  ):
    R_calc = self._compute_window_returns(R, window)
    S, N = R_calc.shape

    if p is None:
      p = np.ones(S) / S
    else:
      p = np.asarray(p)
      p = p[-S:]
      p = p / p.sum()

    if views is not None:
        self.get_views(views, R_calc, p)
    else:
      self.A_eq, self.b_eq, self.A_ineq, self.b_ineq = None, None, None, None

    A_list, b_list = [], []
    k_eq = 0

    if self.A_eq is not None and self.b_eq is not None and len(self.A_eq) > 0:
      A_eq_clean = np.atleast_2d(self.A_eq)
      b_eq_clean = np.atleast_1d(self.b_eq)
      A_list.append(A_eq_clean)
      b_list.append(b_eq_clean)
      k_eq = A_eq_clean.shape[0]

    k_ineq = 0
    if (
          self.A_ineq is not None
          and self.b_ineq is not None
          and len(self.A_ineq) > 0
      ):
      A_ineq_clean = np.atleast_2d(self.A_ineq)
      b_ineq_clean = np.atleast_1d(self.b_ineq)
      A_list.append(A_ineq_clean)
      b_list.append(b_ineq_clean)
      k_ineq = A_ineq_clean.shape[0]


    if A_list:
      A = np.vstack(A_list)
      b = np.concatenate(b_list)
      opt_lambda = self.optimize_lambda(p, A, b, k_eq, k_ineq)
      q = p * np.exp(-opt_lambda @ A)
      q /= np.sum(q)
    else:
      q = p

    mu_post = q @ R_calc.values
    R_dev = R_calc.values - mu_post
    cov_post = (R_dev.T * q) @ R_dev

    mu_post = pd.Series(mu_post, index=R_calc.columns)
    cov_post = pd.DataFrame(
        cov_post,
        index=R_calc.columns,
        columns=R_calc.columns
    )

    return mu_post, cov_post


  def optimize_portfolio(
      self,
      returns,
      views,
      p=None,
      window=1,
      w_prev=None,
      round_weights=False
    ):
    S, N = returns.shape

    p = p if p is not None else np.ones(S)/S
    print("---------------------- Entropy Pooling ----------------------")
    mu_post, cov_post = self.solve_entropy_pooling(returns, views, p, window=window)
    print(f"posterior mean: \n {mu_post}")
    print(f"posterior covariance: \n {cov_post}")

    print("------------------ Optimizing Portfolio w -------------------")
    w_raw = self.optimize_w(returns, mu_post, cov_post, w_prev=w_prev)

    w_opt = self.apply_lot_sizing(w_raw, returns.iloc[-1]) if round_weights else w_raw

    return w_opt, returns.columns





